# Meeting 2026-07-01 — vanilla + cdrmix evotuning sweep (35M)

A new model variant for the DPO low-data learning curve: the **same** single-mask
`cdr_mix` C05 evotuning recipe as `evo_c05_cdrmix_35m`, but seeded from the
**untouched ESM2-35M** (`model.init.source=huggingface`) instead of the general
OAS-evotuned checkpoint. This isolates the effect of the C05 cdrmix fine-tune
from the prior general-OAS evotuning step.

**Selection = validation perplexity only** (`best.pt`, tracked by `best_val_ppl`).
No DMS/Spearman is used to pick the winner. Split is **95/5/0** (no test), with
val-ppl early stopping (`patience=2`).

**Screening sweep**: full factorial, single split salt `oas-v1`, **81 runs** —
`lr ∈ {2e-5,5e-5,8e-5}` × `cdr_mask_prob ∈ {0.5,0.6,0.7}` ×
`cdr_flank ∈ {1,3,5}` × `framework_mask_prob ∈ {0.0,0.05,0.1}`. Scoring disabled.

**Confirmation** (later): top 2–3 combos rerun on extra salts `oas-v2,oas-v3`
with `scoring=c05` (Spearman logged as a diagnostic only — selection still by ppl).


In [ ]:
# === setup + scan the vanilla cdrmix sweep ==========================
%matplotlib inline
import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

def _find_repo_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "conf" / "analysis" / "models.yaml").exists():
            return d
    raise FileNotFoundError("repo root not found (conf/analysis/models.yaml).")

REPO_ROOT = _find_repo_root(Path.cwd())

def _train_dir() -> Path:
    user = os.environ.get("USER", "unknown")
    scratch = os.environ.get("SCRATCH_DIR", f"/cluster/scratch/{user}/protein-design")
    return Path(os.environ.get("TRAIN_DIR", str(Path(scratch) / "train")))

TRAIN_DIR = _train_dir()
TS_RE = re.compile(r"_\d{8}_\d{6}$")  # run dir = <run_name>_<YYYYMMDD_HHMMSS>

# ---- the intended 81-combo screening grid (string literals match the sweep
#      script EXACTLY so run_name reconstruction is byte-identical) -----------
NAME_PREFIX = "vanilla_cdrmix_35m"
LR_GRID    = ["2.0e-5", "5.0e-5", "8.0e-5"]
CMP_GRID   = ["0.5", "0.6", "0.7"]
FLANK_GRID = ["1", "3", "5"]
FMP_GRID   = ["0.0", "0.05", "0.1"]
SCREEN_SALT = "oas-v1"

def run_name_of(lr, cmp, flank, fmp, salt):
    return f"{NAME_PREFIX}_lr{lr}_cmp{cmp}_flank{flank}_fmp{fmp}_{salt}"

def _latest_run_dir(run_name: str, train_dir: Path = TRAIN_DIR):
    """Most-recent <run_name>_<ts>/ that has a summary.json (i.e. finished).
    Returns None if no finished run exists yet."""
    pat = re.compile(r"^" + re.escape(run_name) + r"_\d{8}_\d{6}$")
    best = None  # (mtime, path)
    for p in train_dir.glob(run_name + "_*"):
        if not pat.match(p.name):
            continue
        summ = p / "summary.json"
        if not summ.exists():
            continue
        mt = summ.stat().st_mtime
        if best is None or mt > best[0]:
            best = (mt, p)
    return None if best is None else best[1]

def _best_val_ppl(run_dir: Path):
    """Min val perplexity over per-epoch evals (best.pt selection criterion).
    Reads history.csv (authoritative); falls back to summary.json's final ppl."""
    h = run_dir / "history.csv"
    if h.exists():
        df = pd.read_csv(h)
        if "val/perplexity" in df.columns:
            s = pd.to_numeric(df["val/perplexity"], errors="coerce").dropna()
            if len(s):
                return float(s.min())
    s = run_dir / "summary.json"
    if s.exists():
        try:
            return float(json.loads(s.read_text()).get("val_perplexity"))
        except Exception:
            pass
    return np.nan

def _best_spearman_val(run_dir: Path):
    """Max masked-position M22 val Spearman (only present when scoring=c05).
    Diagnostic only — never used for selection."""
    h = run_dir / "history.csv"
    if h.exists():
        df = pd.read_csv(h)
        if "eval/spearman_M22_val" in df.columns:
            s = pd.to_numeric(df["eval/spearman_M22_val"], errors="coerce").dropna()
            if len(s):
                return float(s.max())
    return np.nan

def scan_grid(lr_g, cmp_g, flank_g, fmp_g, salts):
    """Enumerate the full intended grid; for each combo, attach metrics from the
    most-recent FINISHED run (NaN/None when not yet finished — never an error)."""
    rows = []
    for lr in lr_g:
        for cmp in cmp_g:
            for flank in flank_g:
                for fmp in fmp_g:
                    for salt in salts:
                        rn = run_name_of(lr, cmp, flank, fmp, salt)
                        rd = _latest_run_dir(rn)
                        rows.append(dict(
                            lr=lr, cmp=cmp, flank=flank, fmp=fmp, salt=salt,
                            run_name=rn,
                            finished=rd is not None,
                            best_val_ppl=_best_val_ppl(rd) if rd is not None else np.nan,
                            spearman_val=_best_spearman_val(rd) if rd is not None else np.nan,
                            run_dir=str(rd) if rd is not None else "",
                        ))
    return pd.DataFrame(rows)

grid_df = scan_grid(LR_GRID, CMP_GRID, FLANK_GRID, FMP_GRID, [SCREEN_SALT])
n_total = len(grid_df)
n_done = int(grid_df["finished"].sum())
print(f"TRAIN_DIR = {TRAIN_DIR}")
print(f"Screening grid (salt={SCREEN_SALT}): {n_done}/{n_total} finished, "
      f"{n_total - n_done} missing.")

# cross-reference any sweep manifest(s) under bash_scripts/logs/ (informational)
manifests = sorted((REPO_ROOT / "bash_scripts" / "logs").glob("sweep_evotuning_cdrmix_*.csv"))
if manifests:
    print("\nSweep manifests found:")
    for m in manifests:
        try:
            mdf = pd.read_csv(m)
            print(f"  {m.name}: {len(mdf)} rows")
        except Exception as e:
            print(f"  {m.name}: unreadable ({e})")
else:
    print("\nNo sweep manifest CSVs found under bash_scripts/logs/ "
          "(scanning $TRAIN_DIR directly).")

display(grid_df[grid_df["finished"]].sort_values("best_val_ppl")
        [["lr", "cmp", "flank", "fmp", "best_val_ppl"]].head(10)
        .round(5).reset_index(drop=True))


## Missing experiments — copy-paste relaunch commands

Every screening combo that has **not** finished (no `summary.json` under
`$TRAIN_DIR`) is listed below with the exact command to (re)launch just that one
combo. Commands are printed, **not** executed by the notebook.


In [ ]:
# === missing-combo relaunch commands ================================
missing = grid_df[~grid_df["finished"]]
print(f"{len(missing)} / {len(grid_df)} combos missing.\n")

if missing.empty:
    print("All 81 screening runs finished — nothing to relaunch.")
else:
    print("# --- via the sweep script (one combo each) ---")
    for _, r in missing.iterrows():
        print(
            f"bash_scripts/sweep_evotuning_cdrmix.sh "
            f"--model-preset esm2_35m --freeze-layers 0 --init-source huggingface "
            f"--name-prefix {NAME_PREFIX} "
            f"--lr-grid {r.lr} --cmp-grid {r.cmp} --flank-grid {r.flank} "
            f"--fmp-grid {r.fmp} --salt-grid {r.salt}"
        )
    print("\n# --- equivalently, raw sbatch lines ---")
    for _, r in missing.iterrows():
        print(
            f"sbatch bash_scripts/train.sbatch evotuning_cdrmix "
            f"model=esm2_35m model.init.source=huggingface "
            f"model.freeze_first_n_layers=0 scoring=none "
            f"training.learning_rate={r.lr} data.cdr_mask_prob={r.cmp} "
            f"data.cdr_flank={r.flank} data.framework_mask_prob={r.fmp} "
            f"data.split.salt={r.salt} run_name={r.run_name}"
        )


## Sweep comparison — `best_val_ppl` across the 4 swept dimensions

One heatmap panel per `(cdr_flank, framework_mask_prob)` pair (rows = flank,
cols = fmp). Within each panel: `learning_rate` (y) × `cdr_mask_prob` (x), colored
by `best_val_ppl` (lower = better). Unfinished combos render blank. Uses only
finished runs; the global best combo is printed below.


In [ ]:
# === best_val_ppl heatmaps: lr x cmp, panel per (flank, fmp) ========
fin = grid_df[grid_df["finished"] & grid_df["best_val_ppl"].notna()].copy()

if fin.empty:
    print("No finished runs with a val ppl yet — nothing to plot.")
else:
    vmin, vmax = fin["best_val_ppl"].min(), fin["best_val_ppl"].max()
    nrow, ncol = len(FLANK_GRID), len(FMP_GRID)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow),
                             squeeze=False, constrained_layout=True)
    lrs = list(LR_GRID)          # y axis (top -> bottom as given)
    cmps = list(CMP_GRID)        # x axis
    im = None
    for i, flank in enumerate(FLANK_GRID):
        for j, fmp in enumerate(FMP_GRID):
            ax = axes[i][j]
            M = np.full((len(lrs), len(cmps)), np.nan)
            sub = grid_df[(grid_df.flank == flank) & (grid_df.fmp == fmp)]
            for _, r in sub.iterrows():
                if r.finished and not np.isnan(r.best_val_ppl):
                    M[lrs.index(r.lr), cmps.index(r.cmp)] = r.best_val_ppl
            im = ax.imshow(M, cmap="viridis_r", vmin=vmin, vmax=vmax, aspect="auto")
            ax.set_xticks(range(len(cmps))); ax.set_xticklabels(cmps, fontsize=8)
            ax.set_yticks(range(len(lrs)));  ax.set_yticklabels(lrs, fontsize=8)
            ax.set_title(f"flank={flank}, fmp={fmp}", fontsize=9)
            if i == nrow - 1: ax.set_xlabel("cdr_mask_prob", fontsize=8)
            if j == 0:        ax.set_ylabel("learning_rate", fontsize=8)
            mid = (vmin + vmax) / 2
            for a in range(len(lrs)):
                for b in range(len(cmps)):
                    if np.isnan(M[a, b]):
                        continue
                    ax.text(b, a, f"{M[a, b]:.3f}", ha="center", va="center",
                            fontsize=7, color="white" if M[a, b] > mid else "black")
    if im is not None:
        fig.colorbar(im, ax=axes, shrink=0.6, label="best_val_ppl (lower = better)")
    fig.suptitle("Vanilla cdrmix 35M — val perplexity vs (lr, cmp, flank, fmp)")
    out = REPO_ROOT / "report" / "figures" / "vanilla_cdrmix_35m_sweep_valppl.pdf"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches="tight"); print("Saved:", out)
    plt.show()

    best = fin.sort_values("best_val_ppl").iloc[0]
    print(f"\nCurrent best combo by best_val_ppl ({n_done}/{n_total} finished):")
    print(f"  lr={best.lr}  cmp={best.cmp}  flank={best.flank}  fmp={best.fmp}"
          f"  ->  best_val_ppl={best.best_val_ppl:.5f}")
    print(f"  run_dir = {best.run_dir}")


## Confirmation stage — relaunch top combos on extra salts

Take the top 2–3 hyperparameter combos by `best_val_ppl` and rerun each on
**additional** split salts (`oas-v2,oas-v3`) with `scoring=c05`, to check the
winner isn't an artifact of one split (and to log Spearman as a diagnostic).
`oas-v1` already exists from screening, so we only add `oas-v2,oas-v3`.

`TOP_COMBOS` defaults to the top 3 finished combos — **edit it** to pick exactly
the combos you want to confirm. Commands are printed, not executed.


In [ ]:
# === confirmation-stage launch commands =============================
TOP_K = 3
_fin = grid_df[grid_df["finished"] & grid_df["best_val_ppl"].notna()]
# (lr, cmp, flank, fmp) tuples; edit this list directly to override.
TOP_COMBOS = [
    (r.lr, r.cmp, r.flank, r.fmp)
    for _, r in _fin.sort_values("best_val_ppl").head(TOP_K).iterrows()
]
print("TOP_COMBOS (edit above to change):")
for c in TOP_COMBOS:
    print("  ", c)
print()

if not TOP_COMBOS:
    print("No finished combos yet — run the screening sweep first.")
else:
    print("# confirmation runs (oas-v2 + oas-v3, scoring=c05) — one per combo:")
    for lr, cmp, flank, fmp in TOP_COMBOS:
        print(
            f"bash_scripts/sweep_evotuning_cdrmix.sh "
            f"--model-preset esm2_35m --freeze-layers 0 --init-source huggingface "
            f"--name-prefix {NAME_PREFIX} "
            f"--lr-grid {lr} --cmp-grid {cmp} --flank-grid {flank} --fmp-grid {fmp} "
            f"--salt-grid oas-v2,oas-v3 --scoring c05"
        )


## Final comparison — split uncertainty + ppl↔Spearman diagnostic

For each top candidate, aggregate `best_val_ppl` across all available salts
(`oas-v1` from screening + `oas-v2,oas-v3` from confirmation) → mean ± std.
**Winner = lowest mean val ppl** (selection rule; explicit). Alongside, the
logged M22-val Spearman (from the `scoring=c05` confirmation runs) is reported to
*visually check* how well val-ppl selection tracks the DMS metric — diagnostic
only, never used to pick. Partially-finished confirmation runs are flagged, not
errored.


In [ ]:
# === final comparison across salts (val ppl ± split std) ============
CONFIRM_SALTS = ["oas-v1", "oas-v2", "oas-v3"]

if not TOP_COMBOS:
    print("No TOP_COMBOS — run earlier cells first.")
else:
    recs = []
    for lr, cmp, flank, fmp in TOP_COMBOS:
        cdf = scan_grid([lr], [cmp], [flank], [fmp], CONFIRM_SALTS)
        ppls = {r.salt: r.best_val_ppl for _, r in cdf.iterrows() if r.finished}
        sps  = {r.salt: r.spearman_val for _, r in cdf.iterrows()
                if r.finished and not np.isnan(r.spearman_val)}
        vals = np.array([v for v in ppls.values() if not np.isnan(v)], dtype=float)
        sp_vals = np.array(list(sps.values()), dtype=float)
        recs.append(dict(
            combo=f"lr{lr}_cmp{cmp}_flank{flank}_fmp{fmp}",
            lr=lr, cmp=cmp, flank=flank, fmp=fmp,
            n_salts_done=len(vals),
            salts_done=",".join(sorted(ppls)),
            ppl_mean=float(vals.mean()) if len(vals) else np.nan,
            ppl_std=float(vals.std(ddof=0)) if len(vals) else np.nan,
            ppl_best=float(vals.min()) if len(vals) else np.nan,
            spearman_mean=float(sp_vals.mean()) if len(sp_vals) else np.nan,
        ))
    res = pd.DataFrame(recs).sort_values("ppl_mean")
    display(res.round(5).reset_index(drop=True))

    missing_conf = res[res["n_salts_done"] < len(CONFIRM_SALTS)]
    if not missing_conf.empty:
        print("NOTE: these candidates are missing some confirmation salts "
              "(partial — numbers provisional):")
        for _, r in missing_conf.iterrows():
            have = set(r.salts_done.split(",")) if r.salts_done else set()
            print(f"  {r.combo}: have [{r.salts_done}], "
                  f"missing [{','.join(sorted(set(CONFIRM_SALTS) - have))}]")

    ranked = res[res["ppl_mean"].notna()]
    if not ranked.empty:
        win = ranked.iloc[0]
        print(f"\nFINAL WINNER (rule = lowest MEAN val ppl across salts):")
        print(f"  {win.combo}  ppl_mean={win.ppl_mean:.5f} ± {win.ppl_std:.5f} "
              f"(n_salts={int(win.n_salts_done)})  ppl_best={win.ppl_best:.5f}")
        print(f"  -> register the best.pt of its lowest-ppl salt run in "
              f"conf/analysis/models.yaml (e.g. key 'vanilla_cdrmix_35m').")

        # diagnostic: val ppl vs logged Spearman per candidate
        diag = ranked[ranked["spearman_mean"].notna()]
        if not diag.empty:
            fig, ax = plt.subplots(figsize=(5, 4))
            ax.scatter(diag["ppl_mean"], diag["spearman_mean"], s=60)
            for _, r in diag.iterrows():
                ax.annotate(r.combo, (r.ppl_mean, r.spearman_mean),
                            fontsize=7, xytext=(4, 4), textcoords="offset points")
            ax.set_xlabel("mean best_val_ppl (selection metric)")
            ax.set_ylabel("M22-val Spearman (diagnostic)")
            ax.set_title("val-ppl vs DMS Spearman across top candidates")
            plt.show()
        else:
            print("\n(No scoring=c05 Spearman logged yet — run confirmation stage "
                  "to populate the ppl↔Spearman diagnostic.)")
